# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement deux hypothèses métier sur votre dataset à l'aide de **tests statistiques adaptés** (`scipy.stats`).
2. Construire une **segmentation / typologie non supervisée (K-Means)** combinant agrégation distribuée **PySpark**, transformation d'asymétrie (`log1p`), standardisation (`StandardScaler`) et labellisation métier vers **`/lakehouse/gold/`**.
3. Développer votre **Dashboard Streamlit interactif** dans `app/dashboard.py` (`http://localhost:8501`) en documentant chaque requête SQL et chaque choix statistique.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

ModuleNotFoundError: No module named 'numpy'

## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

**Formulation de vos hypothèses *(À compléter)* :**
- **Test #1 :** $H_0 =$ `Les montants de course (fare_amount) ont la même distribution entre les trajets de nuit (is_night_trip = 1) et les trajets de jour (is_night_trip = 0)` / $H_1 =$ `Les montants de course diffèrent significativement entre la nuit et le jour (Justification du choix du test : Mann-Whitney U (stats.mannwhitneyu), test non paramétrique adapté à deux groupes indépendants sur une variable continue fortement asymétrique avec outliers)` *(Justification du choix du test : `Mann-Whitney U (stats.mannwhitneyu), test non paramétrique adapté à deux groupes indépendants sur une variable continue fortement asymétrique avec outliers).`)*
- **Test #2 :** $H_0 =$ `Il n'existe pas de corrélation monotone entre la distance parcourue (trip_distance) et le montant de la course (fare_amount) ($\rho_s = 0$)` / $H_1 =$ `Il existe une corrélation monotone significative entre la distance parcourue et le montant de la course ($\rho_s \neq 0$)` *(Justification du choix du test : `Corrélation de rangs de Spearman (stats.spearmanr), robuste aux distributions non gaussiennes et aux valeurs extrêmes)`)*

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

SILVER_MAIN_PATH = "hdfs://namenode:9000/lakehouse/silver/nyc_taxi_silver"

df_spark_proj = (
    spark.read.format("delta").load(SILVER_MAIN_PATH)
    .select(
        F.col("fare_amount").cast("double").alias("fare_amount"),
        F.col("trip_distance").cast("double").alias("trip_distance"),
        F.col("is_night_trip").cast("int").alias("is_night_trip"),
    )
    .dropna()
    .limit(20_000)
)

df_sample = df_spark_proj.toPandas()
assert df_sample is not None and len(df_sample) > 0, "TODO : Extrayez les colonnes nécessaires dans `df_sample`."

# Test #1 : Mann-Whitney U (Nuit vs Jour sur fare_amount)
night_fares = df_sample.loc[df_sample["is_night_trip"] == 1, "fare_amount"]
day_fares = df_sample.loc[df_sample["is_night_trip"] == 0, "fare_amount"]
stat_1, p_val_1 = stats.mannwhitneyu(night_fares, day_fares, alternative="two-sided")

print(f"Test #1 (Mann-Whitney U - Nuit vs Jour) : U = {stat_1:.2f}, p-value = {p_val_1:.4e}")
print(f"  Médiane Nuit : {night_fares.median():.2f} | Médiane Jour : {day_fares.median():.2f}")

# Test #2 : Corrélation de Spearman (trip_distance vs fare_amount)
rho, p_val_2 = stats.spearmanr(df_sample["trip_distance"], df_sample["fare_amount"])
print(f"Test #2 (Spearman - Distance vs Tarif) : rho = {rho:.4f}, p-value = {p_val_2:.4e}")

:: loading settings :: url = jar:file:/opt/python/cpython-3.15.2-linux-aarch64-gnu/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7b809187-7c34-4d19-b725-5837ce54c109;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta

Test #1 (Mann-Whitney U - Nuit vs Jour) : U = 39208416.50, p-value = 6.8017e-19
  Médiane Nuit : 10.00 | Médiane Jour : 9.00
Test #2 (Spearman - Distance vs Tarif) : rho = 0.9441, p-value = 0.0000e+00


## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("J4_KMeans_Segmentation")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

SILVER_MAIN_PATH = "hdfs://namenode:9000/lakehouse/silver/nyc_taxi_silver"
GOLD_SEGMENTS_PATH = "hdfs://namenode:9000/lakehouse/gold/nyc_taxi_gold_segments"

# TODO 1 : Agréger les features par entité métier côté PySpark puis convertir en Pandas
df_silver = spark.read.format("delta").load(SILVER_MAIN_PATH)

entities_spark = (
    df_silver
    .filter(F.col("trip_month").isNotNull())
    .groupBy("trip_month")
    .agg(
        F.count("*").cast("double").alias("trip_count"),
        F.sum(F.col("fare_amount").cast("double")).alias("total_fare"),
        F.expr("percentile_approx(cast(fare_amount as double), 0.5)").alias("median_fare"),
        F.avg(F.col("trip_distance").cast("double")).alias("avg_distance"),
    )
    .dropna()
)

entities_pd = entities_spark.toPandas()
assert entities_pd is not None and len(entities_pd) > 0, "TODO : Construisez le DataFrame d'entités agrégées `entities_pd`."

# TODO 2 : Prétraiter (log1p + StandardScaler), entraîner KMeans et nommer métier chaque cluster dans `entities_pd["segment"]`
feature_cols = ["trip_count", "total_fare", "median_fare", "avg_distance"]
X_log = np.log1p(np.clip(entities_pd[feature_cols].values, a_min=0.0, a_max=None))

n_clusters = min(3, len(entities_pd))
if n_clusters >= 2:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_log)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    entities_pd["cluster_id"] = kmeans.fit_predict(X_scaled)
else:
    entities_pd["cluster_id"] = 0

print("=== Médianes par cluster ===")
print(entities_pd.groupby("cluster_id")[feature_cols].median())

cluster_order = (
    entities_pd.groupby("cluster_id")["total_fare"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)
labels_pool = [
    "Fort Volume & Revenu Élevé",
    "Activité Intermédiaire",
    "Volume Modéré / Revenu Plus Faible",
]
segment_map = {cid: labels_pool[i % len(labels_pool)] for i, cid in enumerate(cluster_order)}
entities_pd["segment"] = entities_pd["cluster_id"].map(segment_map)

assert "segment" in entities_pd.columns, "TODO : Ajoutez une colonne lisible `segment` décrivant chaque cluster."

# TODO 3 : Sauvegarder en Delta Lake sur HDFS dans `GOLD_SEGMENTS_PATH`
(
    spark.createDataFrame(entities_pd)
    .coalesce(2)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENTS_PATH)
)
print(f"Table de segmentation enregistrée dans {GOLD_SEGMENTS_PATH} ({len(entities_pd):,} entités) !")

:: loading settings :: url = jar:file:/opt/python/cpython-3.15.2-linux-aarch64-gnu/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b4491c8c-b5b2-4e25-91a6-f7c8e449380b;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta

=== Médianes par cluster ===
            trip_count    total_fare  median_fare  avg_distance
cluster_id                                                     
0           11075287.5  1.374358e+08          9.0      5.034580
1           12136234.0  1.544179e+08          9.5      6.169424
2           12666557.0  1.503172e+08          9.0     13.546626


/opt/python/cpython-3.15.2-linux-aarch64-gnu/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/python/cpython-3.15.2-linux-aarch64-gnu/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/python/cpython-3.15.2-linux-aarch64-gnu/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 's

Table de segmentation enregistrée dans hdfs://namenode:9000/lakehouse/gold/nyc_taxi_gold_segments (4 entités) !


## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

In [ ]:
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path

gold_base = "hdfs://namenode:9000/lakehouse/gold"
gold_root = PathHDFS(gold_base)

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"{gold_base}/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")